# Normative Modeling Pipeline

This notebook implements a normative modeling approach for analyzing brain MRI data, using Bayesian Linear Regression (BLR) with warping functionality.

## Overview

The pipeline consists of several key components:

1. **Data Preparation**
   - Loads preprocessed MRI data
   - Handles missing values using KNN imputation
   - Prepares covariates (age, sex, site)

2. **Model Training**
   - Implements Bayesian Linear Regression with warping
   - Trains separate models for each brain region
   - Optimizes hyperparameters using L-BFGS-B

3. **Model Evaluation**
   - Computes performance metrics:
     - MAE (Mean Absolute Error)
     - RMSE (Root Mean Square Error)
     - Rho (Pearson correlation)
     - SMSE (Standardized Mean Square Error)
     - EXPV (Explained Variance)
     - MSLL (Mean Standardized Log Loss)
     - BIC (Bayesian Information Criterion)

4. **Prediction and Deviation Analysis**
   - Generates predictions for new subjects
   - Calculates Z-scores and deviations
   - Identifies significant deviations from normative ranges

## Usage Instructions

1. Set the correct paths in the first code cell:
   - `root_dir`: Path to project root
   - `docu_dir`: Path to documentation files
   - `data_dir`: Path to preprocessed data

2. Run cells sequentially to:
   - Load and prepare data
   - Train normative models
   - Evaluate model performance
   - Generate predictions and deviations

3. Results will be saved in the specified output directories

## Key Parameters

- `perm`: Number of permutations for cross-validation
- `ROI_list`: List of brain regions to analyze
- `cov`: Covariates to include in the model

## Output Files

The pipeline generates:
- Trained model parameters
- Performance metrics
- Deviation scores

In [ ]:
import os
import pandas as pd
from utils_norm.nm_training import *

root_dir = 'Abosolute path to this project'
docu_dir = os.path.join(root_dir, '1_document')
data_dir = os.path.join(root_dir, '3_rerun_whole_work', '1_data_cleaned')

# NM training

In [2]:
HC_reference = pd.read_csv(os.path.join(data_dir, 'HC_reference_MRI_age.csv'))
HC_reference.rename(columns={'21003-2.0':'age', '31-0.0':'sex', '54-2.0':'site'}, inplace=True)
# Age_group: for train_test_split() only
HC_reference['age_group'] = pd.cut(HC_reference['age'], bins=range(45,85,2), labels=range(46, 84, 2))

HC = pd.read_csv(os.path.join(data_dir, 'HC_MRI_age.csv'))
HC.rename(columns={'21003-2.0':'age', '31-0.0':'sex', '54-2.0':'site'},inplace=True)

pat_data = pd.read_csv(os.path.join(data_dir, 'IBS_All_MRI_age.csv'))
pat_data.rename(columns={'21003-2.0':'age', '31-0.0':'sex', '54-2.0':'site'},inplace=True)

ROI_list = pd.read_csv(os.path.join(docu_dir, 'ROI_IBS.csv'))['ROI'].tolist()
cov = ['age', 'sex', 'site']
perm = 1

In [ ]:
model_dir = os.path.join('3_rerun_whole_work', '2_models_sMRI')
process_and_train_model(root_dir, model_dir, cov, perm, HC_nm_data=HC_reference, HC_data=HC, pat_data=pat_data, pat_name='IBS', ROI_list=ROI_list, sub_name=None, split_data=True)

# NM evaluation

In [4]:
from scipy.stats import levene
import scipy.stats as stats

def t_test_with_assumption(a,b,alternative):
    assump_1 = levene(a, b, center='mean')
    if assump_1[1] > 0.05:
        t_stat, p_value = stats.ttest_rel(a, b, alternative=alternative)
    else:
        t_stat, p_value = stats.wilcoxon(a, b, alternative=alternative)
    return t_stat, p_value

In [ ]:
data_type = ['', '_HC', '_IBS']
file_list = ['SA', 'CT', 'CV']
metrics = ['EV', 'MSLL', 'Skew', 'Kurtosis']  
for file in file_list:
    metrics_dir = os.path.join(root_dir, model_dir, file+'_age_45_85', 'perm_1')
    test_metrics = pd.read_csv(os.path.join(metrics_dir, 'blr_metrics.csv'))
    HC_metrics = pd.read_csv(os.path.join(metrics_dir, 'blr_metrics_HC.csv'))
    IBS_metrics = pd.read_csv(os.path.join(metrics_dir, 'blr_metrics_IBS.csv'))
    filter_idp = test_metrics[test_metrics['pRho_fdr'] > 0.05].index
    test_metrics = test_metrics.drop(filter_idp)
    HC_metrics = HC_metrics.drop(filter_idp)
    IBS_metrics = IBS_metrics.drop(filter_idp)
    for side in ['lh', 'rh']:
        for metric_idx, metric in enumerate(metrics):
            if metric == 'EV':
                alternative = 'greater'
            elif metric == 'MSLL':
                alternative = 'less'
            elif metric == 'Skew':
                alternative = 'less'
            elif metric == 'Kurtosis':
                alternative = 'less'
            if side:
                test_metrics_side = test_metrics[test_metrics['eid'].str.contains(side)][metric]
                HC_metrics_side = HC_metrics[HC_metrics['eid'].str.contains(side)][metric]
                IBS_metrics_side = IBS_metrics[IBS_metrics['eid'].str.contains(side)][metric]
                if metric == 'Skew':
                    test_metrics_side = abs(test_metrics_side)
                    HC_metrics_side = abs(HC_metrics_side)
                    IBS_metrics_side = abs(IBS_metrics_side)
                t_testvsHC, p_testvsHC = t_test_with_assumption(test_metrics_side, HC_metrics_side, alternative)
                t_testvsIBS, p_testvsIBS = t_test_with_assumption(test_metrics_side, IBS_metrics_side, alternative)
                t_HCvsIBS, p_HCvsIBS = t_test_with_assumption(HC_metrics_side, IBS_metrics_side, alternative)
                if p_testvsHC < 0.05:
                    print(f'For {file} {side}, {metric}, test vs. HC: t = {t_testvsHC:.3f}, p = {p_testvsHC:.3f}')
                if p_testvsIBS < 0.05:
                    print(f'For {file} {side}, {metric}, test vs. IBS: t = {t_testvsIBS:.3f}, p = {p_testvsIBS:.3f}')
                if p_HCvsIBS < 0.05:
                    print(f'For {file} {side}, {metric}, HC vs. IBS: t = {t_HCvsIBS:.3f}, p = {p_HCvsIBS:.3f}')